# Krea2 on Kaggle (ComfyUI, GGUF)

Deploy ComfyUI (latest master) + molbal/ComfyUI-GGUF + Krea2 Turbo GGUF Q5_K_M + Qwen3-VL 4B fp8 encoder + Qwen Image VAE, run a 1024x1024 test image.

Why GGUF: the official single-file BF16 is 26.28 GB (over the 20 GB /kaggle/working limit) and the INT8 safetensors route is ~18.98 GB (too tight). The GGUF route is Q5_K_M 8.87 GB + encoder 5.24 GB + VAE 0.25 GB = ~14.37 GB, which fits.

The GGUF loader is molbal/ComfyUI-GGUF (city96's fork is abandoned and does not support Krea2; its architecture allowlist lacks `krea2`). ComfyUI native support for Krea2 is v0.26.0+; the notebook uses latest master (0.44.0 at last check).

Template: official Krea2 Turbo T2I template, expanded to raw nodes (the /prompt API does not accept subgraphs) with the prompt-enhancement (TextGenerate) branch dropped -- the prompt is sent as-is. Sampling: 8 steps, cfg 1, euler, simple. Negative is ConditioningZeroOut (the positive conditioning zeroed out), as in the template. First run is 1024x1024 (<=2 MP); do not try 1696x2112 yet.

Loader choice: `UnetLoaderGGUF` (legacy) is used for the first test -- it needs no Dynamic VRAM feature and is the safe option. `UnetLoaderGGUFDynamicVRAM` is faster on low-VRAM cards (SimpliGen measured ~40 s/step vs ~110 s/step on a 12 GB card) but requires ComfyUI Dynamic VRAM (aimdo) to be enabled, otherwise it raises a RuntimeError. To try it, change GGUF_LOADER in the run cell.

Note: ComfyUI service is started in the run cell via subprocess.Popen (nohup across cells does not survive Kaggle version runs), and stopped when done.

LoRA reserve: the download and run cells each have a LoRA config block (LORA_NAME / LORA_REPO / LORA_STRENGTH, default empty). To add a LoRA later, fill in the filename and repo; the download cell puts it in ComfyUI/models/loras/ and the run cell wires a LoraLoaderModelOnly node (model-only, no CLIP) between the GGUF loader and KSampler. When empty the node is not wired in and the model line is unchanged.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
!if [ -d /kaggle/working/ComfyUI ]; then echo comfyui-exists; else git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI; fi
!cd /kaggle/working/ComfyUI && grep -v -E "^(torch|torchvision|torchaudio)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/comfy_reqs.txt && pip install -q -r /tmp/comfy_reqs.txt
!pip install -q torchsde
!python3 -c "import torchsde; print('torchsde ok')"
# ComfyUI-GGUF (molbal fork -- the one that supports Krea2 GGUF; city96 does not)
!if [ -d /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF ]; then echo gguf-exists; else git clone https://github.com/molbal/ComfyUI-GGUF /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF; fi
!pip install -q "gguf>=0.13.0"
!python3 -c "import gguf; print('gguf ok')"
!ls /kaggle/working/ComfyUI | head -20

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
from huggingface_hub import hf_hub_download
base = "/kaggle/working/ComfyUI/models"
#
# LoRA (optional, reserved for later use): set LORA_NAME to a .safetensors file
# in ComfyUI/models/loras/ and LORA_REPO to the Hugging Face repo hosting it
# (file assumed at the repo root). Leave LORA_NAME = "" (default) to run
# without a LoRA. When set, the file is downloaded here and the run cell wires
# a LoraLoaderModelOnly node into the model line.
LORA_NAME = ""
LORA_REPO = ""
LORA_STRENGTH = 1.0
# Krea2 files (sizes confirmed via HF API):
# - GGUF Q5_K_M 8.87 GB (vantagewithai/Krea-2-Turbo-GGUF, repo root)
# - encoder qwen3vl_4b_fp8_scaled 5.24 GB (Comfy-Org/Krea-2, text_encoders/)
# - VAE qwen_image_vae 0.25 GB (Comfy-Org/Krea-2, vae/)
# Total ~14.37 GB, fits /kaggle/working 20 GB.
# GGUF goes to diffusion_models/ (UnetLoaderGGUF reads unet_gguf = diffusion_models + unet).
p = hf_hub_download(repo_id="vantagewithai/Krea-2-Turbo-GGUF", filename="krea2_turbo-Q5_K_M.gguf", local_dir=base + "/diffusion_models")
print("krea2_turbo-Q5_K_M.gguf", round(os.path.getsize(p) / 1e9, 2), "GB")
for fname in ["text_encoders/qwen3vl_4b_fp8_scaled.safetensors", "vae/qwen_image_vae.safetensors"]:
    p = hf_hub_download(repo_id="Comfy-Org/Krea-2", filename=fname, local_dir=base)
    print(fname, round(os.path.getsize(p) / 1e9, 2), "GB")
if LORA_NAME:
    p = hf_hub_download(repo_id=LORA_REPO, filename=LORA_NAME, local_dir=base + "/loras")
    print("lora", LORA_NAME, round(os.path.getsize(p) / 1e9, 2), "GB")
# Guard: the three files must sit exactly where the loaders look, or fail now.
expected = {
    "diffusion_models": "krea2_turbo-Q5_K_M.gguf",
    "text_encoders": "qwen3vl_4b_fp8_scaled.safetensors",
    "vae": "qwen_image_vae.safetensors",
}
for folder, fname in expected.items():
    fp = os.path.join(base, folder, fname)
    print(folder + "/", fname, "exists:", os.path.exists(fp))
    assert os.path.exists(fp), "missing model file: " + fp
print("all models downloaded")

In [ ]:
import json, time, requests, os, subprocess, sys

BASE = "http://127.0.0.1:8188"
LOG = "/kaggle/working/comfy_server.log"

# Start the ComfyUI server as a child of this cell's python process (nohup across cells does not survive Kaggle version runs).
logf = open(LOG, "wb")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/kaggle/working/ComfyUI",
    stdout=logf, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("server pid:", proc.pid)

up = False
for i in range(36):
    if proc.poll() is not None:
        print("server exited early, code:", proc.returncode)
        break
    try:
        r = requests.get(BASE + "/system_stats", timeout=5)
        if r.status_code == 200:
            up = True
            print("server up after", (i + 1) * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
if not up:
    print("server did not come up; log tail:")
    print(open(LOG, errors="ignore").read()[-3000:])
    proc.terminate()
    raise SystemExit("comfyui server failed to start")

PROMPT = "A photorealistic portrait of a young woman in a sunlit garden, soft morning light, shallow depth of field, 50mm lens"
# GGUF loader: legacy UnetLoaderGGUF for the first test (safe, needs no Dynamic VRAM).
# To try the faster DynamicVRAM loader, change this to "UnetLoaderGGUFDynamicVRAM"
# (requires ComfyUI Dynamic VRAM / aimdo to be enabled, otherwise it raises a RuntimeError).
GGUF_LOADER = "UnetLoaderGGUF"
# LoRA settings are defined in the download cell; default to "no LoRA" if that cell was skipped.
LORA_NAME = globals().get("LORA_NAME", "")
LORA_STRENGTH = globals().get("LORA_STRENGTH", 1.0)

info = requests.get(BASE + "/object_info", timeout=120).json()
have = set(info.keys())
need = [GGUF_LOADER, "CLIPLoader", "VAELoader", "CLIPTextEncode", "ConditioningZeroOut",
        "EmptyLatentImage", "KSampler", "VAEDecode", "SaveImage"]
if LORA_NAME:
    need.append("LoraLoaderModelOnly")
missing = [n for n in need if n not in have]
print("missing nodes:", missing if missing else "none")
if missing:
    proc.terminate()
    raise SystemExit("missing nodes: " + ", ".join(missing))

def node_inputs(class_type, preferred):
    spec = info[class_type]["input"]
    allowed = set(spec.get("required", {})) | set(spec.get("optional", { }))
    return {k: v for k, v in preferred.items() if k in allowed}

# Print the loader + sampler specs to the log so a wrong input name/value is caught here, not after a long run.
for ct in [GGUF_LOADER, "CLIPLoader", "KSampler"]:
    print(ct, "spec:", json.dumps(info[ct]["input"])[:800])

# Raw-node version of the official Krea2 Turbo T2I template (subgraph expanded,
# prompt-enhancement branch dropped, prompt sent as-is). Core pipeline:
# GGUF loader -> CLIPLoader (type krea2) -> VAELoader; CLIPTextEncode ->
# ConditioningZeroOut (negative); EmptyLatentImage 1024x1024; KSampler 8 steps,
# cfg 1, euler, simple; VAEDecode; SaveImage.
nodes = {}
nodes["1"] = {"class_type": GGUF_LOADER, "inputs": node_inputs(GGUF_LOADER, {"unet_name": "krea2_turbo-Q5_K_M.gguf"})}
nodes["2"] = {"class_type": "CLIPLoader", "inputs": node_inputs("CLIPLoader", {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2", "device": "default"})}
nodes["3"] = {"class_type": "VAELoader", "inputs": node_inputs("VAELoader", {"vae_name": "qwen_image_vae.safetensors"})}
nodes["4"] = {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}}
nodes["5"] = {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}}
nodes["6"] = {"class_type": "EmptyLatentImage", "inputs": node_inputs("EmptyLatentImage", {"width": 1024, "height": 1024, "batch_size": 1})}
# LoRA (optional): when LORA_NAME is set, patch the model with LoraLoaderModelOnly
# (model-only, no CLIP) between the GGUF loader and KSampler. When empty, the
# model goes straight from the GGUF loader to KSampler, unchanged.
MODEL_OUT = ["1", 0]
if LORA_NAME:
    nodes["20"] = {"class_type": "LoraLoaderModelOnly", "inputs": node_inputs("LoraLoaderModelOnly", {"model": ["1", 0], "lora_name": LORA_NAME, "strength_model": LORA_STRENGTH})}
    MODEL_OUT = ["20", 0]
nodes["7"] = {"class_type": "KSampler", "inputs": {"model": MODEL_OUT, "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0], "seed": 42, "steps": 8, "cfg": 1, "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0}}
nodes["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}}
nodes["9"] = {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": "krea2_test"}}

# Pre-check: every required input of every node must be present, or fail here with the
# full list (do not let the server reject it one input at a time). Plus a combo-value
# check: where the node definition lists the allowed values, the value we send must be
# one of them (a bad value can be silently dropped and only fail at execution -- the Wan
# V5 lesson). NOTE: a combo's allowed values live in cfg[0] as a LIST; a scalar input
# (STRING/INT/FLOAT) has cfg[0] as a type-name string, so it is skipped -- this is the
# exact check the Qwen notebooks use, and the Krea2 V1 failure was a wrong simplified
# version of it that treated the free-text prompt as a combo value.
problems = []
for nid, node in nodes.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        if isinstance(cfg, list) and cfg and isinstance(cfg[0], list) and isinstance(val, (str, int, float)):
            if val not in cfg[0]:
                problems.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(cfg[0]))
print("missing required inputs:", problems if problems else "none")
if problems:
    proc.terminate()
    raise SystemExit("missing required inputs: " + ", ".join(problems))

r = requests.post(BASE + "/prompt", json={"prompt": nodes}, timeout=180)
print("submit:", r.status_code, r.text[:400])
r.raise_for_status()
pid = r.json()["prompt_id"]
print("prompt_id:", pid)

out_path = "/kaggle/working/krea2_test.png"
t0 = time.time()
done = False
# NOTE: SaveImage writes into ComfyUI/output/, NOT /kaggle/working/. The image must
# be fetched back through the /view API and written to out_path here -- the Krea2 V2
# failure was exactly this step missing (sampling finished in 120 s and node 9 had
# produced krea2_test_00001_.png, but nothing copied it to out_path, so the assert
# below failed). This is the same download step the Qwen/MOSS notebooks use.
while time.time() - t0 < 3600:
    h = requests.get(BASE + "/history/" + pid, timeout=60).json()
    if pid in h:
        st = h[pid].get("status", {})
        sstr = st.get("status_str")
        if sstr == "success":
            outs = h[pid]["outputs"]
            imgs = outs.get("9", {}).get("images", [])
            print("images:", imgs)
            if imgs:
                im = imgs[0]
                q = {"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")}
                data = requests.get(BASE + "/view", params=q, timeout=300).content
                open(out_path, "wb").write(data)
                print("saved", out_path, len(data), "bytes")
                done = True
        break
    time.sleep(30)
print("done:", done, "elapsed:", round(time.time() - t0, 1), "s")
proc.terminate()
print("server stopped")
print("exists:", os.path.exists(out_path), "bytes:", os.path.getsize(out_path) if os.path.exists(out_path) else 0)
assert os.path.exists(out_path), "no output image: " + out_path

In [ ]:
from IPython.display import Image
p = "/kaggle/working/krea2_test.png"
print(p)
display(Image(filename=p))